In [ ]:
import random
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# has to be here for inports to work in vscode
project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from animal_recognition.src.data.augmentations import get_train_transforms, _MEAN, _STD


def denormalize(tensor):

    img = tensor.numpy().transpose(1, 2, 0)

    mean = np.array(_MEAN)
    std = np.array(_STD)

    img = std * img + mean

    img = np.clip(img, 0, 1)
    return img


IMAGE_FOLDER = Path("../../animal_recognition/data/processed/accepted")
NUM_IMAGES = 10
NUM_AUGS = 10
IMAGE_SIZE = 224

valid_exts = {".jpg", ".jpeg", ".png"}
image_paths = [p for p in IMAGE_FOLDER.rglob("*") if p.is_file() and p.suffix.lower() in valid_exts]

if not image_paths:
    print(f"No images found in {IMAGE_FOLDER}")
else:
    print(f"Found {len(image_paths)} images. Generating preview...")

    selected_paths = random.sample(image_paths, min(NUM_IMAGES, len(image_paths)))

    transform = get_train_transforms(image_size=IMAGE_SIZE)

    fig, axes = plt.subplots(
        len(selected_paths), NUM_AUGS + 1, figsize=(3 * (NUM_AUGS + 1), 3 * len(selected_paths))
    )

    if len(selected_paths) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, img_path in enumerate(selected_paths):
        original_img = cv2.imread(str(img_path))
        original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

        ax = axes[i, 0]
        ax.imshow(original_img)
        ax.set_title("Original")
        ax.axis("off")

        for j in range(1, NUM_AUGS + 1):
            augmented_tensor = transform(image=original_img)["image"]

            vis_img = denormalize(augmented_tensor)

            ax = axes[i, j]
            ax.imshow(vis_img)
            ax.set_title(f"Augmented {j}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import sys
from pathlib import Path
import random
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().parent.parent
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from animal_recognition.src.data.dataset import AnimalDataset, CLASSES

data_dir = PROJECT_ROOT / "animal_recognition" / "data" / "processed" / "accepted"
dataset = AnimalDataset(root=data_dir, transform=None)

class_indices = {i: [] for i in range(len(CLASSES))}
for idx, (_, label) in enumerate(dataset.samples):
    if label in class_indices:
        class_indices[label].append(idx)

# 1. Gather up to 5 sampled indices
sampled_data = {}
for label, indices in class_indices.items():
    if not indices:
        continue
    sampled_data[label] = random.sample(indices, min(5, len(indices)))

# 2. Set up the Grid Layout (10 rows, 13 columns)
# Column structure: [Text Left] [5 Images Left] [SPACER] [Text Right] [5 Images Right]
num_rows = 10
images_per_class = 5
num_cols = 13  

# Assign width ratios: give text columns (1.5) and the spacer (1.0) enough room to breathe
width_ratios = [1.5, 1, 1, 1, 1, 1, 1.0, 1.5, 1, 1, 1, 1, 1]

fig, axes = plt.subplots(
    num_rows, 
    num_cols, 
    figsize=(22, 14), 
    gridspec_kw={'width_ratios': width_ratios}, 
    squeeze=False
)
fig.suptitle("Animal Dataset Sample Overview", fontsize=20, y=0.98)

# Hide axes for all subplots beforehand
for r in range(num_rows):
    for c in range(num_cols):
        axes[r, c].axis("off")

# 3. Populate the grid
for i, (label, sampled_indices) in enumerate(sorted(sampled_data.items())):
    class_name = CLASSES[label]
    
    # Placement logic adjusting for the new 13-column layout
    if i < 10:
        row_idx = i
        text_col = 0
        img_start_col = 1
    else:
        row_idx = i - 10
        text_col = 7  # Shifted over to account for the spacer at column 6
        img_start_col = 8
        
    # Draw the label text in its dedicated column axis
    text_ax = axes[row_idx, text_col]
    text_ax.text(1.0, 0.5, f"{class_name}\n({label})", 
                 transform=text_ax.transAxes, fontsize=12, 
                 va='center', ha='right', weight='bold')
        
    # Draw the uncropped images
    for img_idx in range(images_per_class):
        ax = axes[row_idx, img_start_col + img_idx]
        
        if img_idx < len(sampled_indices):
            dataset_idx = sampled_indices[img_idx]
            img_path, img_label = dataset.samples[dataset_idx]
            
            # Load full image without cropping
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)

plt.tight_layout()
plt.show()